# Recon Analysis

Compare two session files for differences

**Before running:** set `DATA_FOLDER` to the folder containing your
session output CSV files. 

In [ ]:
import pandas as pd
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

DATA_FOLDER = Path(r"C:\Git\CandleStateSessionAnalysis\data\recon")
OUTPUT_FOLDER = DATA_FOLDER / "output" 

In [ ]:
import parse_sessions

tables = parse_sessions.parse_sessions(DATA_FOLDER)

# from IPython.display import display
# for table_name, df in tables.items():
#     print(f"### {table_name}")
#     display(df.describe())

trade_signals = tables["trade_signals"]
# out_path = OUTPUT_FOLDER / f"trade_signals.csv"
# trade_signals.to_csv(out_path, index=False)

In [ ]:

key_cols = ['Symbol', 'CandleOpen']
compare_cols = ['Close', 'Volume', 'VolumeRatio', 'VWAP']

pivot = trade_signals.pivot_table(
    index=key_cols,
    columns='SourceFile',
    values=compare_cols,
    aggfunc='first'
)

pivot.columns = [f'{val}__{src}' for val, src in pivot.columns]
pivot = pivot.reset_index()

sources = sorted(trade_signals['SourceFile'].unique())
s1, s2 = sources

pivot['Close_diff']  = pivot[f'Close__{s1}']  - pivot[f'Close__{s2}']
pivot['Volume_diff'] = pivot[f'Volume__{s1}'] - pivot[f'Volume__{s2}']

diffs_only = pivot[(pivot['Close_diff'] != 0) | (pivot['Volume_diff'] != 0)]
diffs_only.head(10)

In [ ]:
import matplotlib.pyplot as plt

trade_signals['CandleOpen'] = pd.to_datetime(trade_signals['CandleOpen'])

compare_cols = ['Close', 'Volume', 'VolumeRatio', 'VWAP']
symbols = sorted(trade_signals['Symbol'].unique())
sources = sorted(trade_signals['SourceFile'].unique())
s1, s2 = sources

fig, axes = plt.subplots(len(symbols), len(compare_cols),
                          figsize=(4*len(compare_cols), 3*len(symbols)),
                          sharex='col')

for i, sym in enumerate(symbols):
    sub = trade_signals[trade_signals['Symbol'] == sym]
    for j, col in enumerate(compare_cols):
        ax = axes[i, j]
        for src, style in zip([s1, s2], ['-', '--']):
            s = sub[sub['SourceFile'] == src].sort_values('CandleOpen')
            ax.plot(s['CandleOpen'], s[col], style,
                    label=src.replace('Session_Live__','').replace('.json',''),
                    linewidth=1)
        if i == 0:
            ax.set_title(col)
        if j == 0:
            ax.set_ylabel(sym)
        ax.tick_params(axis='x', rotation=45, labelsize=6)

axes[0, 0].legend(fontsize=6, loc='upper left')
plt.tight_layout()
plt.show()